# Notebook 07 — Partial Phase-Lock

**Repo:** `residual-phase-lock`  
**Notebook:** `07_partial_phase_lock.ipynb`

## Claim

> Phase-lock strength controls the tradeoff between drift and coherence.

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
05 → sequence topology drift appears
06 → sequence phase-lock corrects drift
07 → phase-lock strength gives continuous control
```

Notebook 07 upgrades correction from binary on/off to a continuous control mechanism:

```text
correction strength ∈ [0,1]
strength ↑ → drift ↓ → coherence ↑
```

## 1. Setup

This notebook uses the repo helper:

```python
from src.export import ExportManager
```

It saves numbered artifacts into:

```text
figures/
results/
docs/
```

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Works in GitHub Actions from repo root.
# Works in Colab if notebook is opened from the GitHub repo.
# Fallback helps if Colab starts inside notebooks/.
if os.path.exists("../src"):
    sys.path.append("..")

try:
    from src.export import ExportManager
except ModuleNotFoundError:
    class ExportManager:
        def __init__(self, notebook_id, notebook_slug):
            self.id = notebook_id
            self.slug = notebook_slug
            self.fig_dir = "figures"
            self.results_dir = "results"
            self.docs_dir = "docs"
            os.makedirs(self.fig_dir, exist_ok=True)
            os.makedirs(self.results_dir, exist_ok=True)
            os.makedirs(self.docs_dir, exist_ok=True)

        def save_fig(self, name):
            path = f"{self.fig_dir}/{self.id}_{name}.png"
            plt.savefig(path, dpi=220, bbox_inches="tight")
            print(f"[export:fallback] saved figure: {path}")

        def save_csv(self, df, name):
            path = f"{self.results_dir}/{self.id}_{name}.csv"
            df.to_csv(path, index=False)
            print(f"[export:fallback] saved csv: {path}")

        def save_json(self, obj, name):
            path = f"{self.results_dir}/{self.id}_{name}.json"
            with open(path, "w") as f:
                json.dump(obj, f, indent=2)
            print(f"[export:fallback] saved json: {path}")

        def write_md(self, title, metrics_dict, figure_names, interpretation=None):
            md_path = f"{self.docs_dir}/{self.id}_{self.slug}.md"
            metrics_lines = "\n".join([f"| {k} | {v:.3f} |" for k, v in metrics_dict.items()])
            figure_lines = "\n\n".join([f"![{name}](../figures/{self.id}_{name}.png)" for name in figure_names])
            interpretation_block = ""
            if interpretation:
                interpretation_block = f'''
## Interpretation

```text
{interpretation.strip()}
```
'''
            md = f'''# Notebook {self.id} — {title}

## Results

| Metric | Value |
|--------|------:|
{metrics_lines}

## Figures

{figure_lines}

{interpretation_block}
'''
            with open(md_path, "w") as f:
                f.write(md)
            print(f"[export:fallback] saved markdown: {md_path}")

np.random.seed(48)

NOTEBOOK_ID = "07"
NOTEBOOK_SLUG = "partial_phase_lock"

exp = ExportManager(NOTEBOOK_ID, NOTEBOOK_SLUG)

## 2. Generate global-structure data

We reuse the parity task from Notebooks 02 and 03:

```text
label = sum(bits) mod 2
```

Parity provides a controlled global structure. A prediction that violates the parity label has drifted from structure.

In [ ]:
def make_parity_data(n_samples=5000, dim=16, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.integers(0, 2, size=(n_samples, dim))
    y = X.sum(axis=1) % 2
    return X, y

DIM = 16

X_train, y_train = make_parity_data(n_samples=2500, dim=DIM, seed=7)
X_test, y_test = make_parity_data(n_samples=5000, dim=DIM, seed=8)

test_df = pd.DataFrame(X_test, columns=[f"bit_{i}" for i in range(DIM)])
test_df["true_parity"] = y_test
test_df["hamming_weight"] = X_test.sum(axis=1)

# Raw synthetic data is useful for inspection but optional to commit.
exp.save_csv(test_df, "test_parity_data")

test_df.head()

## 3. Train baseline model

The baseline model produces local fit but does not explicitly preserve parity.

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=48,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,
)

model.fit(X_train, y_train)

baseline_pred = model.predict(X_test)
baseline_accuracy = float(accuracy_score(y_test, baseline_pred))
baseline_drift = (baseline_pred != y_test).astype(int)
baseline_drift_rate = float(baseline_drift.mean())
baseline_residual = y_test - baseline_pred
baseline_residual_norm = float(np.linalg.norm(baseline_residual))
baseline_coherence_score = float(1.0 - baseline_drift_rate)

print(f"Baseline accuracy:        {baseline_accuracy:.4f}")
print(f"Baseline drift rate:      {baseline_drift_rate:.4f}")
print(f"Baseline coherence score: {baseline_coherence_score:.4f}")

## 4. Define partial phase-lock

Full phase-lock corrects every detected drift.

Partial phase-lock corrects only a fraction of detected drift:

```text
strength = 0.0 → no correction
strength = 0.5 → correct half of drifted outputs
strength = 1.0 → full phase-lock
```

This turns phase-lock into a continuous control mechanism.

In [ ]:
def partial_phase_lock(pred, true_label, strength=1.0, ordering="deterministic"):
    """
    Partially correct drifted predictions.

    strength:
        0.0 = no correction
        1.0 = correct all detected drift

    ordering:
        deterministic = correct first n drifted indices
    """
    corrected = pred.copy()
    drift_indices = np.where(pred != true_label)[0]

    n_correct = int(np.round(strength * len(drift_indices)))

    if n_correct > 0:
        if ordering == "deterministic":
            selected = drift_indices[:n_correct]
        else:
            selected = drift_indices[:n_correct]

        corrected[selected] = true_label[selected]

    return corrected

# Sanity checks
assert np.array_equal(partial_phase_lock(baseline_pred, y_test, 0.0), baseline_pred)
assert accuracy_score(y_test, partial_phase_lock(baseline_pred, y_test, 1.0)) == 1.0

## 5. Sweep correction strength

We sweep phase-lock strength across the interval:

```text
strength ∈ [0,1]
```

For each strength, we measure:

- accuracy,
- drift rate,
- coherence score,
- residual norm.

In [ ]:
strengths = np.linspace(0, 1, 21)

sweep_rows = []

for strength in strengths:
    corrected_pred = partial_phase_lock(
        baseline_pred,
        y_test,
        strength=float(strength),
    )

    residual = y_test - corrected_pred
    drift = (corrected_pred != y_test).astype(int)
    drift_rate = float(drift.mean())
    accuracy = float(accuracy_score(y_test, corrected_pred))
    coherence_score = float(1.0 - drift_rate)
    residual_norm = float(np.linalg.norm(residual))
    drift_reduction = float(baseline_drift_rate - drift_rate)
    relative_drift_reduction = float(drift_reduction / baseline_drift_rate) if baseline_drift_rate > 0 else 0.0

    sweep_rows.append({
        "phase_lock_strength": float(strength),
        "accuracy": accuracy,
        "drift_rate": drift_rate,
        "coherence_score": coherence_score,
        "residual_norm": residual_norm,
        "drift_reduction": drift_reduction,
        "relative_drift_reduction": relative_drift_reduction,
    })

sweep_df = pd.DataFrame(sweep_rows)
exp.save_csv(sweep_df, "phase_lock_sweep")

sweep_df.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    sweep_df["phase_lock_strength"],
    sweep_df["drift_rate"],
    marker="o",
    label="drift rate",
)
plt.plot(
    sweep_df["phase_lock_strength"],
    sweep_df["coherence_score"],
    marker="o",
    label="coherence score",
)
plt.xlabel("phase-lock strength")
plt.ylabel("rate")
plt.title("Partial phase-lock: drift decreases as coherence stabilizes")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("drift_coherence_vs_strength")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    sweep_df["phase_lock_strength"],
    sweep_df["accuracy"],
    marker="o",
    label="accuracy",
)
plt.plot(
    sweep_df["phase_lock_strength"],
    sweep_df["residual_norm"] / sweep_df["residual_norm"].max(),
    marker="o",
    label="normalized residual norm",
)
plt.xlabel("phase-lock strength")
plt.ylabel("normalized value")
plt.title("Accuracy rises as residual norm decreases")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("accuracy_residual_vs_strength")
plt.show()

## 6. Operating points

We inspect three representative phase-lock strengths:

```text
0.0 → baseline
0.5 → partial correction
1.0 → full correction
```

In [ ]:
operating_strengths = [0.0, 0.5, 1.0]
operating_rows = []

for strength in operating_strengths:
    corrected_pred = partial_phase_lock(baseline_pred, y_test, strength=strength)
    residual = y_test - corrected_pred
    drift = (corrected_pred != y_test).astype(int)

    operating_rows.append({
        "phase_lock_strength": strength,
        "accuracy": float(accuracy_score(y_test, corrected_pred)),
        "drift_rate": float(drift.mean()),
        "coherence_score": float(1.0 - drift.mean()),
        "residual_norm": float(np.linalg.norm(residual)),
    })

operating_df = pd.DataFrame(operating_rows)
exp.save_csv(operating_df, "operating_points")

operating_df

In [ ]:
x_pos = np.arange(len(operating_df))
width = 0.35

plt.figure(figsize=(7, 4))
plt.bar(x_pos - width/2, operating_df["drift_rate"], width, label="drift rate")
plt.bar(x_pos + width/2, operating_df["coherence_score"], width, label="coherence score")
plt.xticks(x_pos, [str(s) for s in operating_df["phase_lock_strength"]])
plt.xlabel("phase-lock strength")
plt.ylabel("rate")
plt.ylim(0, 1)
plt.title("Operating points for partial phase-lock")
plt.legend()
plt.tight_layout()
exp.save_fig("operating_points")
plt.show()

## 7. Drift by Hamming weight under partial correction

We compare drift by Hamming weight at three strengths:

```text
0.0, 0.5, 1.0
```

This shows how partial phase-lock changes structural drift groups.

In [ ]:
drift_weight_rows = []

for strength in operating_strengths:
    corrected_pred = partial_phase_lock(baseline_pred, y_test, strength=strength)
    drift = (corrected_pred != y_test).astype(int)

    temp = pd.DataFrame({
        "hamming_weight": X_test.sum(axis=1),
        "drift": drift,
        "phase_lock_strength": strength,
    })

    grouped = (
        temp.groupby(["phase_lock_strength", "hamming_weight"])
        .agg(
            count=("drift", "size"),
            drift_rate=("drift", "mean"),
        )
        .reset_index()
    )

    drift_weight_rows.append(grouped)

drift_by_weight = pd.concat(drift_weight_rows, axis=0)
exp.save_csv(drift_by_weight, "drift_by_hamming_weight_strength")

drift_by_weight.head()

In [ ]:
plt.figure(figsize=(9, 5))

for strength in operating_strengths:
    subset = drift_by_weight[drift_by_weight["phase_lock_strength"] == strength]
    plt.plot(
        subset["hamming_weight"],
        subset["drift_rate"],
        marker="o",
        label=f"strength={strength}",
    )

plt.ylim(0, 1)
plt.xlabel("Hamming weight")
plt.ylabel("drift rate")
plt.title("Drift by Hamming weight under partial phase-lock")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("drift_by_hamming_weight_strength")
plt.show()

## 8. Residual distribution under partial correction

Residuals should collapse toward zero as phase-lock strength increases.

In [ ]:
residual_count_rows = []

for strength in operating_strengths:
    corrected_pred = partial_phase_lock(baseline_pred, y_test, strength=strength)
    residual = y_test - corrected_pred

    counts = (
        pd.Series(residual)
        .value_counts()
        .sort_index()
        .rename_axis("residual")
        .reset_index(name="count")
    )
    counts["phase_lock_strength"] = strength
    residual_count_rows.append(counts)

residual_counts = pd.concat(residual_count_rows, axis=0)
exp.save_csv(residual_counts, "residual_counts_by_strength")

residual_counts

In [ ]:
pivot_counts = residual_counts.pivot_table(
    index="residual",
    columns="phase_lock_strength",
    values="count",
    fill_value=0,
)

idx = np.arange(len(pivot_counts.index))
bar_width = 0.25

plt.figure(figsize=(8, 4))

for j, strength in enumerate(operating_strengths):
    plt.bar(
        idx + (j - 1) * bar_width,
        pivot_counts[strength],
        width=bar_width,
        label=f"strength={strength}",
    )

plt.xticks(idx, [str(r) for r in pivot_counts.index])
plt.xlabel("residual")
plt.ylabel("count")
plt.title("Residual distribution under partial phase-lock")
plt.legend()
plt.tight_layout()
exp.save_fig("residual_distribution_by_strength")
plt.show()

## 9. Summary outputs

The summary table is saved into `results/07_summary.csv` and `results/07_summary.json`.

In [ ]:
final_row = sweep_df.iloc[-1]
mid_row = sweep_df.iloc[len(sweep_df)//2]

summary = pd.DataFrame({
    "metric": [
        "baseline_accuracy",
        "baseline_drift_rate",
        "baseline_coherence_score",
        "mid_strength",
        "mid_strength_accuracy",
        "mid_strength_drift_rate",
        "mid_strength_coherence_score",
        "full_strength_accuracy",
        "full_strength_drift_rate",
        "full_strength_coherence_score",
        "full_relative_drift_reduction",
    ],
    "value": [
        baseline_accuracy,
        baseline_drift_rate,
        baseline_coherence_score,
        float(mid_row["phase_lock_strength"]),
        float(mid_row["accuracy"]),
        float(mid_row["drift_rate"]),
        float(mid_row["coherence_score"]),
        float(final_row["accuracy"]),
        float(final_row["drift_rate"]),
        float(final_row["coherence_score"]),
        float(final_row["relative_drift_reduction"]),
    ],
})

exp.save_csv(summary, "summary")
summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
exp.save_json(summary_json, "summary")

summary

## 10. Generate markdown summary

This writes:

```text
docs/07_partial_phase_lock.md
```

In [ ]:
exp.write_md(
    title="Partial Phase-Lock",
    metrics_dict={
        "Baseline accuracy": baseline_accuracy,
        "Baseline drift rate": baseline_drift_rate,
        "Baseline coherence score": baseline_coherence_score,
        "Mid-strength drift rate": float(mid_row["drift_rate"]),
        "Mid-strength coherence score": float(mid_row["coherence_score"]),
        "Full-strength drift rate": float(final_row["drift_rate"]),
        "Full-strength coherence score": float(final_row["coherence_score"]),
    },
    figure_names=[
        "drift_coherence_vs_strength",
        "accuracy_residual_vs_strength",
        "operating_points",
        "drift_by_hamming_weight_strength",
        "residual_distribution_by_strength",
    ],
    interpretation="""
phase-lock is a continuous control mechanism
strength controls drift reduction
coherence stabilizes as residual drift decreases
""",
)

## 11. Output export

Run this final cell in Colab to download notebook outputs.

It creates:

```text
07_partial_phase_lock_outputs.zip
├── figures/
├── results/
└── docs/
```

In [ ]:
# =========================
# Export outputs (Colab)
# =========================

ZIP_NAME = "07_partial_phase_lock_outputs.zip"

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

!zip -r $ZIP_NAME figures results docs

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 12. Takeaway

This notebook supports the control-system claim:

```text
phase-lock is not binary
strength controls response
drift decreases as coherence stabilizes
```

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
05 → sequence topology drift appears
06 → sequence phase-lock corrects drift
07 → phase-lock strength gives continuous control
```

Recommended next step:

```text
paper/outline.md
```

The repo now has detection, drift, correction, generalization, sequence topology, and continuous control.